# Conv

In [3]:
import numpy as np

In [4]:
def conv_forward_naive(x, w, b, conv_param):
    N, C, H, W = x.shape
    F, C, HH, WW = w.shape
    pad, stride = conv_param["pad"], conv_param["stride"]
    X = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)), "constant")
    
    H_out = 1 + int((H + 2 * pad - HH) / stride)
    W_out = 1 + int((W + 2 * pad - WW) / stride)
    out = np.zeros((N, F, H_out, W_out))
    
    for n in range(N):
        for f in range(F):
            for i in range(H_out):
                for j in range(W_out):
                    data = X[n, :, i * stride: i * stride + HH, j * stride: j * stride + WW].reshape(1, -1)
                    filt = w[f, :, :, :].reshape(-1, 1)
                    out[n, f, i, j] = data.dot(filt) + b[f]
                    
    cache = (x, w, b, conv_param)
    return out, cache

In [2]:
def conv_backward_naive(dout, cache):
    x, w, b, conv_param = cache
    N, F, H_out, W_out = dout.shape
    N, C, H, W = x.shape
    F, C, HH, WW = w.shape
    pad, stride = conv_param["pad"], conv_param["stride"]
    X = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)), "constant")
    dw = np.zeros_like(w)
    dX = np.zeros_like(X)
    
    for n in range(N):
        for f in range(F):
            for i in range(H_out):
                for j in range(W_out):
                    dX[n, :, i * stride: i * stride + HH, j * stride: j * stride + WW] += w[f] * dout[n, f, i, j]
                    dw[f] += X[n, :, i * stride: i * stride + HH, j * stride: j * stride + WW] * dout[n, f, i, j]
                    
    db = np.sum(dout, axis=(0, 2, 3))
    dx = dX[:, :, pad: -pad, pad: -pad]
    return dx, dw, db

In [8]:
x = np.arange(16).reshape(1, 1, 4, 4)
print("x: \n", x)
w = np.ones((1, 1, 3, 3))
print("w: \n", w)
b = np.ones((1,))
print("b: \n", b)
conv_param = {"pad": 1, "stride": 1}
out, cache = conv_forward_naive(x, w, b, conv_param)
print("out: \n", out)

x: 
 [[[[ 0  1  2  3]
   [ 4  5  6  7]
   [ 8  9 10 11]
   [12 13 14 15]]]]
w: 
 [[[[1. 1. 1.]
   [1. 1. 1.]
   [1. 1. 1.]]]]
b: 
 [1.]
out: 
 [[[[11. 19. 25. 19.]
   [28. 46. 55. 40.]
   [52. 82. 91. 64.]
   [43. 67. 73. 51.]]]]
